[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/geo-incrementality/blob/main/notebooks/geo_incrementality_codealong.ipynb)

# 📍 Geo Incrementality Testing: A Code-Along
### Did the campaign *cause* those sales, or would they have happened anyway?

Ad platforms report conversions. They can't tell you how many of those people would have bought anyway. **Incrementality** is the share of results the campaign actually *caused*, and a **geo holdout test** is one of the most trusted ways to measure it: turn the campaign on in some markets, hold others out, and compare.

Doing that credibly takes four steps, and you'll build every one from scratch:

| Step | Question it answers | Method you'll code |
|---|---|---|
| 1. Assignment | Are treatment and control genuinely comparable? | Matched-pair randomization |
| 2. Measurement | What would treated markets have done without the campaign? | Difference-in-differences, synthetic control |
| 3. Inference | Is the result real, or noise? | Placebo tests |
| 4. Power | Could this test detect the lift we expect, before we spend a dollar? | Simulated power analysis |

Then you'll write the readout a CFO actually cares about, and prove the whole method is honest by running it 100 times.

**Why simulated data?** In real life nobody knows the true lift, so you can never check your answer. Here we *choose* the truth, hide it, and see whether the method finds it.

⏱️ About 45–60 minutes. Everything runs on Colab's free tier, with no API keys needed. Look for 🧠 checkpoints along the way: try answering before you open the hint.

## 1. Setup

Everything we need ships with Colab: `numpy`, `pandas`, `scipy` (for the synthetic control solver) and `matplotlib`.

`FAST_MODE` cuts the number of simulations so cells run in seconds while you're learning. Turn it off for final, publication-quality numbers. Those match the GitHub repo exactly.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import nnls

FAST_MODE = True     # False = full simulation counts (matches the repo's published results)
N_NULL   = 100 if FAST_MODE else 300   # placebo tests per analysis
N_POWER  = 80  if FAST_MODE else 200   # simulated tests per lift in the power analysis
N_REPS   = 30  if FAST_MODE else 100   # full repeated experiments in the validation step

NAVY, BLUE, ORANGE, GREEN, GRAY = "#1F3864", "#2E86AB", "#F18F01", "#6A994E", "#999999"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
money = lambda x: f"${x:,.0f}"
print("FAST_MODE:", FAST_MODE)

## 2. The core idea: the missing counterfactual

Every causal question compares two worlds:

- **Actual:** what treated markets did *with* the campaign. We observe this.
- **Counterfactual:** what those same markets would have done *without* it. We can never observe this directly.

$$\text{incremental revenue} = \text{actual} - \text{counterfactual}$$

$$\text{lift} = \frac{\text{incremental revenue}}{\text{counterfactual}} \qquad \text{iROAS} = \frac{\text{incremental revenue}}{\text{campaign spend}}$$

The whole craft of incrementality testing is building a **believable counterfactual**. Control markets are our window into it: if they behaved like the treated markets before the test, they tell us what the treated markets would have done during it.

> 🧠 **Check your understanding:** A campaign's dashboard shows $1M in attributed revenue. A geo test finds $300K incremental. Is the dashboard lying?

<details><summary>Show answer</summary>

Not exactly. The dashboard counts every conversion it can link to an ad, including people who would have bought anyway, like loyal customers or people already searching for the brand. The geo test counts only revenue that wouldn't have existed without the campaign. Both numbers can be correct; they answer different questions. Budget decisions should use the incremental one.

</details>

## 3. Build a world with a known truth

We'll simulate **40 geos** (think DMAs) with weekly revenue. To be realistic, and to give the methods something to struggle with, each geo gets:

- **Its own size**, from about $20K to $300K a week. Real markets differ wildly.
- **Shared national seasonality**, with each geo following it a bit more or less strongly.
- **Its own slight trend.**
- **AR(1) noise.** Good and bad weeks cluster, because each week's surprise carries half of last week's. This is what makes real data hard: noise that isn't independent from week to week fools naive statistics.

In [ ]:
def simulate_panel(n_geos=40, pre_weeks=26, seed=11):
    # Baseline weekly revenue per geo before any test runs (long format).
    rng = np.random.default_rng(seed)
    weeks = pre_weeks + 12                      # room for a test window
    t = np.arange(weeks)
    national = 1 + 0.12 * np.sin(2 * np.pi * (t + 8) / 52) + 0.05 * np.sin(2 * np.pi * t / 13)
    rows = []
    for g in range(n_geos):
        size = rng.lognormal(np.log(80_000), 0.6)
        trend = rng.normal(0, 0.002)
        sensitivity = rng.normal(1.0, 0.15)    # how strongly this geo follows national seasonality
        eps = np.zeros(weeks)
        for i in range(1, weeks):               # AR(1) noise: good and bad weeks cluster
            eps[i] = 0.5 * eps[i - 1] + rng.normal(0, 0.04)
        revenue = size * (1 + sensitivity * (national - 1)) * (1 + trend * t) * (1 + eps)
        rows.append(pd.DataFrame({"geo": f"G{g:02d}", "week": t, "revenue": revenue}))
    return pd.concat(rows, ignore_index=True)

def to_wide(panel, value="revenue"):
    # One row per week, one column per geo: the shape every estimator works on.
    return panel.pivot(index="week", columns="geo", values=value).sort_index()

PRE_WEEKS, TEST_WEEKS, PLANNED_LIFT = 26, 6, 0.06
panel = simulate_panel(n_geos=40, pre_weeks=PRE_WEEKS, seed=11)
wide_pre = to_wide(panel).iloc[:PRE_WEEKS]      # only history: this is all we'd have before a test
wide_pre.iloc[:5, :6].round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for g in wide_pre.columns:
    ax.plot(wide_pre.index, wide_pre[g] / 1e3, color=BLUE, alpha=0.35, lw=1)
ax.set_title("26 weeks of history for 40 geos: different sizes, shared seasonality, clustered noise")
ax.set_xlabel("week"); ax.set_ylabel("weekly revenue ($K)"); ax.set_yscale("log")
plt.show()

## 4. Step 1: Assignment with matched pairs

If we flipped a coin for each geo, we might accidentally put most of the big markets in treatment. That imbalance adds noise and can even look like an effect.

**Matched pairs fix this.** Sort the geos by pre-period revenue, pair neighbors (the two smallest, the next two, and so on), and randomly treat one of each pair. Every treated geo has a similar-sized partner in control. We keep the benefit of randomization, which protects against hidden bias, while reducing the noise from luck.

In [ ]:
def matched_pair_assignment(wide_pre, seed=0):
    # Pair geos by pre-period revenue and randomly treat one of each pair.
    rng = np.random.default_rng(seed)
    order = wide_pre.sum().sort_values().index.tolist()
    return [order[i + rng.integers(0, 2)] for i in range(0, len(order) - 1, 2)]

treated = matched_pair_assignment(wide_pre, seed=42)
controls = [g for g in wide_pre.columns if g not in treated]
print(f"{len(treated)} treated, {len(controls)} control")
print(f"Pre-period revenue  treated: {money(wide_pre[treated].sum().sum())}   control: {money(wide_pre[controls].sum().sum())}")

> 🧠 **Check your understanding:** Why randomize *within* pairs instead of just picking the 20 geos that look most similar to the rest?

<details><summary>Show answer</summary>

Hand-picking lets hidden factors sneak in. You might unconsciously choose markets where the sales team is stronger, or where a competitor is weaker. Randomizing within pairs balances the things we can see, like size, through pairing, and protects against the things we can't see through chance.

</details>

## 5. Step 2: Two ways to build the counterfactual

Both estimators take the same inputs: weekly revenue for every geo, the treated list, when the test starts, and how long it runs. Both return a weekly **counterfactual** for the treated group.

### Estimator A: Difference-in-differences (ratio form)

**Assumption:** treated geos keep the same *ratio* to control geos that they had before the test. If treated geos were 0.98x control before, then without the campaign they'd still be 0.98x control during the test. Any extra is the effect.

It's simple and transparent, but it weights every control geo equally, even ones that behave nothing like the treated group.

In [ ]:
def _split(wide, treated, start, weeks):
    controls = [g for g in wide.columns if g not in treated]
    idx = np.arange(len(wide))
    return controls, idx < start, (idx >= start) & (idx < start + weeks)

def diff_in_diff(wide, treated, start, weeks):
    controls, pre, test = _split(wide, treated, start, weeks)
    y = wide[treated].sum(axis=1).to_numpy()       # treated total, every week
    c = wide[controls].sum(axis=1).to_numpy()      # control total, every week
    ratio = y[pre].sum() / c[pre].sum()            # the pre-test relationship
    return y, c * ratio, None, pre, test           # counterfactual = control x that ratio

### Estimator B: Synthetic control

**Idea:** instead of averaging all controls equally, build a custom **blend** of control geos that tracks the treated group as closely as possible before the test. If something like 16x G09 plus 8x G30 plus a little of a few others reproduces the treated group's pre-test history week by week, it's a strong stand-in for what would have happened next. The multipliers are large only because 20 treated markets are being rebuilt from individual control markets.

We find those weights by solving a least-squares problem, **constrained to be non-negative** (`nnls`). Non-negativity matters: it stops the model from "subtracting" markets, which would let it fit noise and produce wild counterfactuals.

In [ ]:
def synthetic_control(wide, treated, start, weeks):
    # Non-negative weights on control geos, fit to the treated total before the test.
    controls, pre, test = _split(wide, treated, start, weeks)
    y = wide[treated].sum(axis=1).to_numpy()
    X = wide[controls].to_numpy()
    w, _ = nnls(X[pre], y[pre])                    # fit ONLY on pre-test weeks
    return y, X @ w, pd.Series(w, index=controls), pre, test

def estimate(wide, treated, start, weeks, method="synthetic_control"):
    fn = {"synthetic_control": synthetic_control, "diff_in_diff": diff_in_diff}[method]
    y, cf, weights, pre, test = fn(wide, treated, start, weeks)
    incremental = float((y[test] - cf[test]).sum())
    return {"method": method,
            "incremental_revenue": incremental,
            "lift": incremental / float(cf[test].sum()),
            "pre_fit_mape": float(np.mean(np.abs(y[pre] - cf[pre]) / y[pre])),   # how well we tracked history
            "actual": y, "counterfactual": cf, "weights": weights}

> 🧠 **Check your understanding:** Synthetic control fits its weights only on the pre-test weeks. What would go wrong if it also used the test weeks?

<details><summary>Show answer</summary>

It would bend the counterfactual toward the treated group's actual test-period revenue, which already contains the campaign's effect. That would absorb part of the lift into the 'what would have happened' line and shrink the estimate toward zero. The counterfactual must never see the outcome it's trying to predict.

</details>

## 6. Step 3: Is it real? Placebo tests

Suppose synthetic control estimates a 4% lift. Is that the campaign, or could noise produce 4% on its own?

**Placebo test:** take *historical* data where no campaign ran, pretend the last six weeks were a test, randomly assign fake treatment groups, and run the exact same analysis hundreds of times. Every estimate you get is pure noise. That collection is the **null distribution**: what this design produces when nothing is happening.

- **p-value:** the share of placebo estimates at least as large as ours. If only 0.3% of no-effect tests look this big, noise is an unlikely explanation.
- **Confidence interval:** if placebo errors spread from −3% to +3% around zero, the true lift is plausibly within that same range of our estimate. We invert the placebo errors.

Note what we *don't* need: no normality assumption and no textbook formula. The data tells us its own noise level, AR(1) clustering included.

In [ ]:
def null_distribution(wide_pre, test_weeks, n_sims=300, method="synthetic_control", seed=0):
    # Estimated lift when there is NO effect, across many random fake assignments.
    rng = np.random.default_rng(seed)
    fake_start = len(wide_pre) - test_weeks
    return np.array([estimate(wide_pre, matched_pair_assignment(wide_pre, seed=int(rng.integers(1e9))),
                              fake_start, test_weeks, method)["lift"] for _ in range(n_sims)])

def placebo_inference(observed_lift, null, level=0.9):
    # Two-sided p-value and a confidence interval built by inverting the placebo errors.
    p = float((np.sum(np.abs(null) >= abs(observed_lift)) + 1) / (len(null) + 1))
    lo_q, hi_q = np.quantile(null, [(1 - level) / 2, 1 - (1 - level) / 2])
    return {"p_value": p, "ci_low": observed_lift - hi_q, "ci_high": observed_lift - lo_q, "level": level}

null_sc = null_distribution(wide_pre, TEST_WEEKS, n_sims=N_NULL)
print(f"Placebo estimates (no real effect): mean {null_sc.mean():+.2%}, middle 90% from "
      f"{np.quantile(null_sc, .05):+.2%} to {np.quantile(null_sc, .95):+.2%}")

> 🧠 **Check your understanding:** Why does the p-value formula add 1 to both the numerator and the denominator?

<details><summary>Show answer</summary>

It counts the observed result as one more possible outcome under the null. That prevents ever reporting p = 0, which would claim certainty that a finite number of placebo tests can't support. With 300 placebos, the smallest possible p-value is 1/301, about 0.003.

</details>

## 7. Step 4: Power analysis, before spending a dollar

In real life this step comes **first**. We built the estimators earlier only because power analysis uses them.

The question: *if the campaign truly lifts revenue by X%, how often would this test detect it?* We answer it with history alone:

1. Take the pre-period data and a fake test window.
2. Randomly assign treatment and **inject** a known lift, say 3%.
3. Run the analysis and check whether the estimate clears the significance threshold set by the placebo distribution.
4. Repeat many times. The detection rate is the **power** at that lift.

The **minimum detectable effect (MDE)** is the smallest lift we'd catch at least 80% of the time. If you expect a 3% lift and the MDE is 6%, the test is a coin flip. Don't launch it as designed: add markets, run longer, or spend more.

In [ ]:
def power_curve(wide_pre, test_weeks, lifts=(0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08),
                n_sims=200, alpha=0.05, method="synthetic_control", seed=0):
    rng = np.random.default_rng(seed)
    null = null_distribution(wide_pre, test_weeks, n_sims, method, seed)
    threshold = np.quantile(np.abs(null), 1 - alpha)          # what noise alone rarely exceeds
    fake_start = len(wide_pre) - test_weeks
    rows = []
    for lift in lifts:
        hits = 0
        for _ in range(n_sims):
            tr = matched_pair_assignment(wide_pre, seed=int(rng.integers(1e9)))
            w = wide_pre.copy()
            w.loc[w.index[fake_start:], tr] *= 1 + lift          # inject a known lift
            hits += estimate(w, tr, fake_start, test_weeks, method)["lift"] > threshold
        rows.append({"true_lift": lift, "power": hits / n_sims})
    return pd.DataFrame(rows)

def minimum_detectable_effect(curve, target_power=0.8):
    ok = curve[curve.power >= target_power]
    return float(ok.true_lift.min()) if len(ok) else None

curve = power_curve(wide_pre, TEST_WEEKS, n_sims=N_POWER)
mde = minimum_detectable_effect(curve)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve.true_lift * 100, curve.power * 100, marker="o", color=NAVY, lw=2)
ax.axhline(80, color=GRAY, ls="--", lw=1)
if mde: ax.axvline(mde * 100, color=ORANGE, ls="--", label=f"MDE at 80% power: {mde:.0%}")
ax.axvline(PLANNED_LIFT * 100, color=GREEN, ls=":", lw=2, label=f"Expected lift: {PLANNED_LIFT:.0%}")
ax.set_xlabel("True lift (%)"); ax.set_ylabel("Chance of detecting it (%)")
ax.set_title(f"Power analysis: 40 geos, {TEST_WEEKS}-week test"); ax.legend(frameon=False); plt.show()
curve

> 🧠 **Check your understanding:** Name three levers that lower the MDE, and the cost of each.

<details><summary>Show answer</summary>

(1) **More geos:** more matched pairs average away noise, but you hold out more of the market from the campaign. (2) **A longer test:** more weeks of signal, but it costs more time and the risk of something else changing mid-test grows. (3) **More spend in treated geos:** a bigger true lift is easier to see, but the test costs more. A fourth, subtler lever is **better market matching**, which lowers noise for free.

</details>

## 8. Launch the test and read it

The design passes, so we launch. In the simulation, treated geos spend 3.5% of their baseline revenue on the campaign and get a true lift around 6%, with a little geo-to-geo variation, because no campaign lands evenly. The `GeoTruth` object is the answer key. Pretend you can't see it until the end.

In [ ]:
@dataclass
class GeoTruth:
    lift: float
    incremental_revenue: float
    incremental_spend: float
    @property
    def iroas(self): return self.incremental_revenue / self.incremental_spend

def apply_test(panel, treated, start, weeks, lift=0.06, spend_share=0.035, seed=3):
    rng = np.random.default_rng(seed)
    df = panel[panel.week < start + weeks].copy()
    df["spend"] = 0.0
    in_test = df.geo.isin(treated) & df.week.between(start, start + weeks - 1)
    geo_lift = {g: max(0.0, rng.normal(lift, lift * 0.3)) for g in treated}
    base = df.loc[in_test, "revenue"].to_numpy()
    lifts = df.loc[in_test, "geo"].map(geo_lift).to_numpy()
    df.loc[in_test, "revenue"] = base * (1 + lifts)
    df.loc[in_test, "spend"] = base * spend_share
    return df, GeoTruth(float((base * lifts).sum() / base.sum()), float((base * lifts).sum()),
                        float((base * spend_share).sum()))

observed, truth = apply_test(panel, treated, PRE_WEEKS, TEST_WEEKS, lift=PLANNED_LIFT)
wide = to_wide(observed)
spend = observed.spend.sum()

results = {}
for method in ["synthetic_control", "diff_in_diff"]:
    est = estimate(wide, treated, PRE_WEEKS, TEST_WEEKS, method)
    null = null_sc if method == "synthetic_control" else null_distribution(wide_pre, TEST_WEEKS, N_NULL, method)
    inf = placebo_inference(est["lift"], null)
    cf_test = est["counterfactual"][PRE_WEEKS:PRE_WEEKS + TEST_WEEKS].sum()
    results[method] = {**est, **inf, "null": null, "iroas": est["incremental_revenue"] / spend,
                       "iroas_low": inf["ci_low"] * cf_test / spend, "iroas_high": inf["ci_high"] * cf_test / spend}

pd.DataFrame({m: {"estimated lift": f"{r['lift']:.2%}",
                  "90% interval": f"{r['ci_low']:.2%} to {r['ci_high']:.2%}",
                  "p-value": round(r["p_value"], 3),
                  "incremental revenue": money(r["incremental_revenue"]),
                  "pre-test fit error": f"{r['pre_fit_mape']:.2%}"} for m, r in results.items()}).T

In [ ]:
sc = results["synthetic_control"]
weeks = np.arange(len(sc["actual"]))
fig, ax = plt.subplots(figsize=(10, 4))
ax.axvspan(PRE_WEEKS - .5, PRE_WEEKS + TEST_WEEKS - .5, color=ORANGE, alpha=.08, label="Test window")
ax.plot(weeks, sc["actual"], color=NAVY, lw=2, label="Treated geos (actual)")
ax.plot(weeks, sc["counterfactual"], color=BLUE, lw=2, ls="--", label="Synthetic control (what would have happened)")
ax.set_title(f"Treated geos vs. synthetic control (pre-test fit error {sc['pre_fit_mape']:.1%})")
ax.yaxis.set_major_formatter(lambda v, _: f"${v/1e6:,.2f}M"); ax.set_xlabel("week")
ax.legend(frameon=False, fontsize=8, loc="lower left"); plt.show()

ctrl = [g for g in wide.columns if g not in treated]
share = (wide[ctrl] * sc["weights"]).sum() / sc["counterfactual"].sum()
print(f"Synthetic control uses {int((sc['weights'] > 0).sum())} of {len(ctrl)} control geos. Share of the blend:")
print(share[share > 0].sort_values(ascending=False).head(8).map("{:.0%}".format).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(sc["null"] * 100, bins=30, color=GRAY, alpha=.7, label="Placebo tests (no real effect)")
ax.axvline(sc["lift"] * 100, color=NAVY, lw=2.5, label=f"Observed lift {sc['lift']:.1%} (p = {sc['p_value']:.3f})")
ax.set_xlabel("estimated lift (%)"); ax.set_ylabel("placebo tests")
ax.set_title("Is the result real? Observed lift vs. placebo tests"); ax.legend(frameon=False, fontsize=8); plt.show()

> 🧠 **Check your understanding:** Synthetic control's pre-test fit error is about half of diff-in-diff's. Why does that matter for the test period?

<details><summary>Show answer</summary>

The counterfactual during the test is only as trustworthy as its fit before the test. A model that tracks history tightly has less room to drift when it extrapolates into the test window, so its estimate carries less noise. That's also why its placebo distribution is narrower, which shows up as a lower p-value.

</details>

## 9. The readout a CFO cares about

Now reveal the answer key and translate lift into money.

In [ ]:
print(f"TRUTH: lift {truth.lift:.2%} | incremental revenue {money(truth.incremental_revenue)} | iROAS {truth.iroas:.2f}\n")
for m, r in results.items():
    print(f"{m:<18} incremental {money(r['incremental_revenue']):>10} | "
          f"iROAS {r['iroas']:.2f} (90% interval {r['iroas_low']:.2f} to {r['iroas_high']:.2f})")

Read that carefully, because this is the skill that separates a good analyst from a dashboard:

- **Did the campaign *drive* sales?** Yes, clearly. The lift interval sits well above zero, and the p-value is far below 0.05 (0.003 with full simulations; `FAST_MODE` can't go below about 0.01 because it runs only 100 placebos).
- **Did it *pay*?** Not proven. The iROAS interval dips just below 1.0, so at 90% confidence we can't rule out that each dollar returned less than a dollar.

Those are two different questions, and a credible readout separates them. The right recommendation isn't "it worked, scale it." It's "it works; extend or enlarge the test to confirm profitability before scaling." The fix is more evidence, not a rounder number.

## 10. Prove the method is honest: run it again and again

One good result could be luck. The real test of a method is whether its **intervals mean what they say**: a 90% interval should contain the truth about 90% of the time across many experiments.

So we rerun the *entire* pipeline (new market data, new assignment, new campaign, new placebo distribution) many times and count. This is called **calibration**. It's the strongest evidence you can put in a portfolio, because it shows the method is honest, not just lucky once.

⏱️ About 15 seconds in `FAST_MODE`, about a minute with the full 100 runs.

In [ ]:
reps = []
for s in range(N_REPS):
    p = simulate_panel(n_geos=40, pre_weeks=PRE_WEEKS, seed=1000 + s)
    wp = to_wide(p).iloc[:PRE_WEEKS]
    tr = matched_pair_assignment(wp, seed=s)
    obs, t = apply_test(p, tr, PRE_WEEKS, TEST_WEEKS, lift=PLANNED_LIFT, seed=s)
    e = estimate(to_wide(obs), tr, PRE_WEEKS, TEST_WEEKS)
    inf = placebo_inference(e["lift"], null_distribution(wp, TEST_WEEKS, n_sims=150, seed=s))
    reps.append({"true": t.lift, "est": e["lift"], "covered": inf["ci_low"] <= t.lift <= inf["ci_high"],
                 "significant": inf["p_value"] < 0.05})
reps = pd.DataFrame(reps)
print(f"{N_REPS} repeated tests")
print(f"  average estimate {reps.est.mean():.2%} vs. average truth {reps.true.mean():.2%}")
print(f"  90% interval contained the truth: {reps.covered.mean():.0%} of tests (target: 90%)")
print(f"  effect detected (p < 0.05):       {reps.significant.mean():.0%} of tests")

> 🧠 **Check your understanding:** With the full 100 runs, coverage lands at 90%. (With only 30 runs in `FAST_MODE` it bounces around more.) What would 60% coverage tell you? What about 100%?

<details><summary>Show answer</summary>

**60%** means the intervals are too narrow: the method is overconfident and would mislead decision-makers. That's exactly what happens if you use a textbook formula that ignores the clustered (AR(1)) noise. **100%** means the intervals are too wide: honest but wasteful, so you'd need bigger, costlier tests than necessary. Close to the nominal level is the goal.

</details>

## 11. ✍️ Your turn

1. **Underpowered test:** set `PLANNED_LIFT = 0.03` and re-run sections 8–10. How often is the effect detected now? Does the power curve from section 7 predict it?
2. **Trade time for markets:** set `TEST_WEEKS = 3`, then `TEST_WEEKS = 10`, and compare the MDE. What does that cost the business?
3. **Spillover:** after `apply_test`, add 2% to every *control* geo during the test window, simulating people in control markets seeing the campaign. Which way does the estimate move, and why?
4. **A local shock:** add a 15% jump to one treated geo in test week 3, like a local event or a competitor closing. Does synthetic control get fooled? How would you guard against it?
5. **Connect to MMM:** a marketing mix model estimated this channel's ROI at 2.4. What would you tell the modeling team, and how should they use this test?

---

### Where this fits

Incrementality tests are the ground truth that keeps a marketing mix model honest. An MMM estimates every channel at once from observational data; a geo test measures one channel causally. Best practice is to use test results to calibrate the MMM. See the companion `mmm-from-first-principles` project.

**Next steps** if you want to go further: augmented synthetic control, Bayesian structural time series (Google's CausalImpact), and optimized market selection.

*Built by Na'im McKee. Simulated data with a known truth; the methods apply directly to real geo-level sales data.*